# 05 — Design Your Algorithm

You have exact (02), the measurement loop (03), and fuzzy (04). Now we **design** the full pipeline: **combining** exact and fuzzy, fuzzy first + exact last, and a **blended** score so you have one threshold to tune. Same measurement loop—evaluate and compare. Blocking (06) and deduplication (07) are optional next steps.

**Back:** [04 Fuzzy Matching](04_fuzzy_matching.ipynb) · **Next:** [06 Blocking](06_blocking.ipynb)

## 1. Load data and ground truth

Same data as 02–04: load **raw** entity resolution data, **standardize** it (aligned schema, `email_clean`, `full_name`), and build the matcher. We use the same 500×500, 30-pair ground truth so every design choice in this notebook is comparable to what you saw in the measurement loop (03) and fuzzy (04).

In [1]:
import sys
from pathlib import Path
import polars as pl
from matcher import Matcher, FuzzyMatcher, SimpleEvaluator

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = (
        Path.cwd()
        if (Path.cwd() / "tutorial_data").exists()
        else Path.cwd().parent.parent / "docs" / "tutorial"
    )
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_entity_resolution, standardize_for_matching

left_raw, right_raw, ground_truth = load_entity_resolution()
left, right = standardize_for_matching(left_raw, right_raw)
matcher = Matcher(left=left, right=right, left_id="id", right_id="id")
print(f"Left: {left.shape[0]} rows, Right: {right.shape[0]} rows, Ground truth: {ground_truth.shape[0]} pairs")
print(left.select(["first_name", "last_name", "full_name"]).head(3))

Left: 500 rows, Right: 500 rows, Ground truth: 30 pairs
shape: (3, 3)
┌────────────┬───────────┬─────────────┐
│ first_name ┆ last_name ┆ full_name   │
│ ---        ┆ ---       ┆ ---         │
│ str        ┆ str       ┆ str         │
╞════════════╪═══════════╪═════════════╡
│ Alice      ┆ Smith     ┆ Alice Smith │
│ Bob        ┆ Jones     ┆ Bob Jones   │
│ Carol      ┆ White     ┆ Carol White │
└────────────┴───────────┴─────────────┘


## 2. The improvement loop

This is the loop you'll use from here on: (1) run matcher with your current rules and blocking, (2) evaluate against ground truth, (3) change one thing (rules, blocking key, fuzzy threshold, or blend cutoff), (4) re-run and compare. Repeat until precision and recall are where you need them. No magic—just measure, tune, compare.

## 3. Exact + fuzzy combo

Often you want both: exact matches (e.g. email) *and* fuzzy matches (e.g. name) that catch pairs where email is missing or different. Run exact and fuzzy separately, merge the pair sets, dedupe by (id, id_right), then evaluate the combined set. You'll see whether the combo improves recall without hurting precision too much.

In [2]:
exact_results = matcher.match(on="email_clean")
fuzzy_results = matcher.match(on=["full_name"], matching_algorithm=FuzzyMatcher(threshold=0.85))

exact_pairs = exact_results.matches.select(["id", "id_right"]).unique()
fuzzy_pairs = fuzzy_results.matches.select(["id", "id_right"]).unique()
combo_pairs = pl.concat([exact_pairs, fuzzy_pairs]).unique(subset=["id", "id_right"])

metrics_exact = exact_results.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
metrics_combo = SimpleEvaluator().evaluate(combo_pairs, ground_truth, left_id_col="id", right_id_col="id_right")
print("Exact only: ", f"precision={metrics_exact['precision']:.2%}, recall={metrics_exact['recall']:.2%}")
print("Exact+fuzzy:", f"precision={metrics_combo['precision']:.2%}, recall={metrics_combo['recall']:.2%}")

Exact only:  precision=100.00%, recall=33.33%
Exact+fuzzy: precision=9.39%, recall=96.67%


## 4. Fuzzy first + exact last

Fuzzy on **full_name** can pair "Christopher Lee" with "Christine James"—similar first name, different last name—so you get false positives. A better pattern: fuzzy on **first_name** only, then keep pairs where **last_name** matches exactly. There's no single built-in rule for that; you run `match(on=["first_name"], matching_algorithm=FuzzyMatcher(...))` and filter to `last_name == last_name_right`, then wrap in `MatchResults` if you want to call `.evaluate()`. Below we use the **same** data already loaded (entity resolution from 02–04) and compare fuzzy full_name vs fuzzy first + exact last.

In [3]:
from matcher.results import MatchResults

# Fuzzy on full_name (as in 04)
fuzzy_full = matcher.match(on=["full_name"], matching_algorithm=FuzzyMatcher(threshold=0.85))
m_full = fuzzy_full.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")

# Fuzzy first + exact last: match(on=["first_name"], ...) with FuzzyMatcher, then keep only exact last_name
fuzzy_first = matcher.match(on=["first_name"], matching_algorithm=FuzzyMatcher(threshold=0.85))
filtered = fuzzy_first.matches.filter(pl.col("last_name") == pl.col("last_name_right"))
results_first_exact_last = MatchResults(filtered, original_left=matcher.left, source=matcher)
m_first_last = results_first_exact_last.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")

print("Fuzzy full_name (0.85):     ", f"precision={m_full['precision']:.2%}, recall={m_full['recall']:.2%}, F1={m_full['f1']:.2%}")
print("Fuzzy first + exact last:   ", f"precision={m_first_last['precision']:.2%}, recall={m_first_last['recall']:.2%}, F1={m_first_last['f1']:.2%}")

Fuzzy full_name (0.85):      precision=7.89%, recall=80.00%, F1=14.37%
Fuzzy first + exact last:    precision=100.00%, recall=70.00%, F1=82.35%


## 5. Blended algorithm: one score per pair

Instead of "exact OR fuzzy" as separate outputs, you can **blend** them: assign a score to each source (e.g. exact = 1.0, fuzzy = confidence), group by pair and take the max (or another aggregation), then apply a single cutoff. One score per pair, one threshold to tune—simpler to reason about and to ship.

A practical **algorithm** is: **(1) fixed (exact) name or email** — match on `email_clean` or on exact `first_name` + `last_name`; give those pairs score 1.0. **(2) Fuzzy first + exact last** — run `match(on=["first_name"], matching_algorithm=FuzzyMatcher(...))`, keep only pairs where `last_name == last_name_right`, and use the confidence as their score. Combine both into one set of pairs, take the max score per pair, and apply a single cutoff (e.g. 0.85). That way you get high precision from exact identifiers and from exact last name, and you still recover name variants (Jon/Jonathan) via fuzzy first name. Below we implement that blend.

In [11]:
# (1) Fixed name or email — exact rules, score 1.0
exact = matcher.match(on="email_clean").refine(on=["first_name", "last_name"])
exact_df = exact.matches.select(["id", "id_right"]).unique().with_columns(pl.lit(1.0).alias("score"))

# (2) Fuzzy first + exact last — confidence as score
fuzzy_first = matcher.match(on=["first_name"], matching_algorithm=FuzzyMatcher(threshold=0.5))
fuzzy_first_exact_last = fuzzy_first.matches.filter(pl.col("last_name") == pl.col("last_name_right"))
fuzzy_df = fuzzy_first_exact_last.select(["id", "id_right", "confidence"]).with_columns(pl.col("confidence").alias("score"))

# Combine, max score per pair, single cutoff
all_pairs = pl.concat([
    exact_df.select(["id", "id_right", "score"]),
    fuzzy_df.select(["id", "id_right", "score"])
])
blended = all_pairs.group_by(["id", "id_right"]).agg(pl.col("score").max().alias("blended_score"))
final = blended.filter(pl.col("blended_score") >= 0.85)

metrics = SimpleEvaluator().evaluate(final, ground_truth, left_id_col="id", right_id_col="id_right")

print("Blended algorithm (exact name/email + fuzzy first & exact last, cutoff 0.85):")
print(f"  precision = {metrics['precision']:.2%}")
print(f"  recall    = {metrics['recall']:.2%}")
print(f"  F1        = {metrics['f1']:.2%}")
print(f"  pairs above cutoff = {final.height}")
print()
print("This combination gives both high precision and high recall.")
print()
print("Sample of pairs with blended score:")
sample = final.join(left.select(["id", "full_name"]), on="id", how="left")
sample = sample.join(right.select(pl.col("id").alias("id_right"), pl.col("full_name").alias("full_name_right")), on="id_right", how="left")
sample.select(["id", "id_right", "blended_score", "full_name", "full_name_right"]).head(8).sort("blended_score", descending=False)

Blended algorithm (exact name/email + fuzzy first & exact last, cutoff 0.85):
  precision = 100.00%
  recall    = 90.00%
  F1        = 94.74%
  pairs above cutoff = 27

This combination gives both high precision and high recall.

Sample of pairs with blended score:


id,id_right,blended_score,full_name,full_name_right
str,str,f64,str,str
"""left_30""","""right_30""",0.888889,"""Alexander Hill""","""Alex Hill"""
"""left_22""","""right_22""",0.925926,"""Katherine Jones""","""Catherine Jones"""
"""left_1""","""right_1""",1.0,"""Alice Smith""","""Alice Smith"""
"""left_7""","""right_7""",1.0,"""Grace Kim""","""Grace Kim"""
"""left_17""","""right_17""",1.0,"""Daniel Lewis""","""Dan Lewis"""
"""left_9""","""right_9""",1.0,"""Iris Chen""","""Iris Chen"""
"""left_2""","""right_2""",1.0,"""Bob Jones""","""Bob Jones"""
"""left_4""","""right_4""",1.0,"""Dave Brown""","""Dave Brown"""


The blended algorithm (exact name or email plus fuzzy first & exact last) gives **both high precision and high recall** on this data: exact rules and exact last name keep false positives low, while fuzzy first name recovers the name-variant pairs. We used 0.85 as a single cutoff; in practice you'd sweep cutoffs and compare precision/recall as in 03. You can try different cutoffs (e.g. 0.82 vs 0.85) or a weighted blend instead of max; the loop is the same—evaluate on the same ground truth and compare.

---

**Closing the loop.** In this notebook you **designed** a full pipeline on the same data as 02–04: **(1)** exact + fuzzy combo (merge pair sets, dedupe), **(2)** fuzzy first + exact last (better precision than fuzzy full_name), **(3)** blended algorithm—exact name or email (score 1.0) plus fuzzy first & exact last (confidence as score), one max per pair, one cutoff. That gives you a single algorithm to tune and ship. Same measurement habit: change the cutoff or the rules, re-run, evaluate, compare.

**Next:** [06 Blocking](06_blocking.ipynb) — restrict comparisons by key when data is large (optional). [07 Deduplication](07_deduplication.ipynb) — same ideas on a single table with `Deduplicator`.